In [1]:
!pip install transformers
!pip install -U bitsandbytes>=0.46.1

In [2]:
import os
import csv
import torch
from PIL import Image
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

# --- Configuration ---
IMAGE_FOLDER = "/kaggle/input/datasets/davidmagdy/market-collage-qwen/collages_qwen" 
OUTPUT_CSV = "/kaggle/working/qwen3_descriptions.csv"

INSTRUCTION_PROMPT = """You are an expert in person re-identification. 
Analyze this collage and provide a highly detailed, comma-separated description of the person. 
Focus strictly on: gender, clothing types, clothing colors, footwear, and any distinguishing accessories (like backpacks, glasses, or hats). 

CRITICAL INSTRUCTIONS: 
- Output ONLY the comma-separated description. 
- Do NOT include any introductory text, pre-text, or greetings.
- Do NOT include any concluding text or follow-up questions.
- Start your response immediately with the first descriptive word."""

def main():
    print("Loading Qwen3-VL-8B on a P100 GPU...")
    
    model_id = "Qwen/Qwen3-VL-8B-Instruct"
    
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16
    )
    
    # CRITICAL FIX: device_map="cuda:0" forces the entire model onto one GPU.
    # This completely eliminates the multi-GPU crashing bugs you experienced.
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        model_id,
        device_map="cuda:0",
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
    )
    
    processor = AutoProcessor.from_pretrained(model_id)
    
    with open(OUTPUT_CSV, mode='w', newline='', encoding='utf-8') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["person_id", "description"])
        
        for filename in sorted(os.listdir(IMAGE_FOLDER)):
            if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
                
            person_id = os.path.splitext(filename)[0]
            image_path = os.path.join(IMAGE_FOLDER, filename)
            
            try:
                # Load image
                image = Image.open(image_path).convert("RGB")
                
                # Qwen-specific message formatting
                messages = [
                    {"role": "user", "content": [
                        {"type": "image", "image": image},
                        {"type": "text", "text": INSTRUCTION_PROMPT}
                    ]}
                ]
                
                # Apply template and move strictly to cuda:0
                inputs = processor.apply_chat_template(
                    messages, 
                    tokenize=True, 
                    add_generation_prompt=True, 
                    return_dict=True, 
                    return_tensors="pt"
                ).to("cuda:0")
                
                # Generate description
                output_ids = model.generate(
                    **inputs, 
                    max_new_tokens=150, 
                    temperature=0.2, 
                    do_sample=True
                )
                
                # Slice off the prompt tokens to get just the generated text
                generated_ids = [
                    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, output_ids)
                ]
                
                description = processor.batch_decode(
                    generated_ids, 
                    skip_special_tokens=True, 
                    clean_up_tokenization_spaces=True
                )[0].strip()
                
                # Final cleanup to ensure no AI filler text slipped through
                common_prefixes = ["Here is the description:", "Sure,", "Description:"]
                for prefix in common_prefixes:
                    if description.lower().startswith(prefix.lower()):
                        description = description[len(prefix):].strip()
                        
                print(f"Processed ID: {person_id} | Output: {description[:50]}...")
                writer.writerow([person_id, description])
                
            except Exception as e:
                print(f"Error processing {filename}: {e}")

    print(f"Finished successfully! Data saved to {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

Loading Qwen3-VL-8B on a P100 GPU...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Processed ID: 0001 | Output: female, sleeveless dress, light beige, barefoot, l...
Processed ID: 0002 | Output: male, red t-shirt, blue denim shorts, dark-colored...
Processed ID: 0003 | Output: female, black short-sleeved top, olive green short...
Processed ID: 0004 | Output: male, red t-shirt, black shorts, black sneakers, b...
Processed ID: 0005 | Output: male, short-sleeved t-shirt, striped pattern, dark...
Processed ID: 0006 | Output: male, white t-shirt, light gray shorts, brown sand...
Processed ID: 0007 | Output: male, t-shirt, gray, shorts, dark blue, sneakers, ...
Processed ID: 0008 | Output: female, white t-shirt with panda graphic, green sh...
Processed ID: 0009 | Output: female, white t-shirt, dark blue shorts, white sne...
Processed ID: 0010 | Output: female, red short-sleeved top, blue ruffled skirt,...
Processed ID: 0011 | Output: female, red short-sleeved t-shirt, light gray shor...
Processed ID: 0012 | Output: male, short-sleeved t-shirt, dark shorts, black sn...
Proc